## Load Image location metadata

In [1]:
import pandas as pd
import sys

# Load the metadata
path = "../chestx-ray/VinDr-PCXR"
df = pd.read_csv(path + "/train/image_labels_train.csv")
df2 = pd.read_csv(path + "/image_labels_test.csv")

train_classdist = df.sum()
test_classdist = df2.sum()

df.head(2)
print("train class distance:", train_classdist)
print("test class distance:", test_classdist)


train class distance: image_id                    6cb53aff85c71b98ad13d67a131708c640414c05687cdb...
rad_ID                      R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3...
No finding                                                             5143.0
Bronchitis                                                              842.0
Brocho-pneumonia                                                        545.0
Other disease                                                           412.0
Bronchiolitis                                                           497.0
Situs inversus                                                           11.0
Pneumonia                                                               392.0
Pleuro-pneumonia                                                          6.0
Diagphramatic hernia                                                      3.0
Tuberculosis                                                             14.0
Congenital emphysema                      

## Create download txt

In [2]:
pathologies = ["No finding"]

df_download = df[df[pathologies].sum(axis=1) > 0]

print(len(df_download))


5143


In [3]:
# Create the list of URLs
base_url = "https://physionet.org/files/vindr-pcxr/1.0.0/train/"
with open(path + '/download_list_full.txt', 'w') as f:
    for img_id in df_download['image_id']:
        f.write(f"{base_url}{img_id}.dicom\n")

print(f"Done! Created 'download_list.txt' with {len(df_download)} targeted URLs.")

Done! Created 'download_list.txt' with 5143 targeted URLs.


## Execute Downloading of download list

In [ ]:
import os
import requests
from tqdm import tqdm

# Deine Daten
USERNAME = 'kallepalle'
PASSWORD = 'Appl1edD33p' # Dein echtes Passwort nutzen
LIST_FILE = '../chestx-ray/VinDr-PCXR/download_list_full.txt'
OUTPUT_DIR = f'../chestx-ray/VinDr-PCXR/train_{pathologies[0].replace(" ", "_")}'

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

session = requests.Session()

# 1. Login-Seite aufrufen, um den CSRF-Token zu erhalten
login_url = "https://physionet.org/login/"
get_response = session.get(login_url)
csrftoken = session.cookies.get('csrftoken')

# 2. Login-Daten mit CSRF-Token vorbereiten
login_data = {
    'username': USERNAME,
    'password': PASSWORD,
    'csrfmiddlewaretoken': csrftoken,
    'next': '/content/vindr-pcxr/1.0.0/'
}

# 3. Einloggen (Referer-Header ist oft wichtig)
session.post(login_url, data=login_data, headers={'Referer': login_url})

# 4. Download-Liste einlesen
with open(LIST_FILE, 'r') as f:
    urls = [line.strip() for line in f if line.strip()]

print(f"Starte sicheren Download von {len(urls)} Dateien...")

for url in tqdm(urls):
    filename = os.path.join(OUTPUT_DIR, os.path.basename(url))
    
    # Prüfen, ob Datei schon existiert (spart Zeit bei Abbruch)
    if os.path.exists(filename):
        continue

    response = session.get(url, stream=True)
    
    if response.status_code == 200:
        with open(filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024*1024): # 1MB Chunks
                f.write(chunk)
    elif response.status_code == 403:
        print(f"\nFehler 403 bei {url}. Login fehlgeschlagen oder DUA nicht aktiv.")
        break 
    else:
        print(f"\nFehler {response.status_code} bei {url}")

print("\nFertig!")

Starte sicheren Download von 5143 Dateien...


 83%|████████▎ | 4255/5143 [03:41<29:14,  1.98s/it]  